# **Load Datset**
Load Data Set of Ev Charging locations in NSW Australia. This dataset is downloaded from Government of Australia open data hub Transport...!

In [356]:
import pandas as pd
df = pd.read_csv('/content/ev_20251216.csv')
display(df.head())

,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,NaN,NaN,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.262242,150.890139,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,NaN,NaN,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.811004,150.849597,Blacktown City Council,2766,Existing Fast Chargers
2,NaN,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.511874,151.669395,Central Darling Shire Council,2350,TfNSW Regional
3,NaN,NaN,"1 Balfour St, Sydney, 2070",Chargefox,7,AC,22 kW,-33.774101,151.167035,Ku-ring-gai Council,2070,Existing Destination Chargers
4,NaN,NaN,"1 Bay Ln, Byron Bay, 2481",Tesla,2,AC,19 kW,-28.641819,153.613633,Byron Shire Council,2481,Existing Destination Chargers


# Data Insights & Overview

Information about dataset

1.   Info/Shape
2.   description/ Summary


In [357]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1958 entries, 0 to 1957
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OBJECTID         121 non-null    float64
 1   Station_name     520 non-null    object 
 2   Station_address  1958 non-null   object 
 3   Operator         1958 non-null   object 
 4   Number_of_plugs  1958 non-null   int64  
 5   Charger_Type     1958 non-null   object 
 6   Charger_rating   1958 non-null   object 
 7   Latitude         1958 non-null   float64
 8   Longitude        1958 non-null   float64
 9   LGANAME          1837 non-null   object 
 10  PCODE            1837 non-null   object 
 11  Source           1837 non-null   object 
dtypes: float64(3), int64(1), object(8)
memory usage: 183.7+ KB


In [358]:
df.shape

(1958, 12)

In [359]:
df.size

23496

In [360]:
X= len(df)

# DATA Quality
Delete All Column which has alot Null Values to ensure the data quality.

A variable is created X, which is the total number of rows in  the data.

Threshold is formulated as
Deleted Columns

if Null values > X(Total length of data) - 10 % of X(Total length of data)

Else Leave the Same

In [361]:
def delete_sparse_columns(dataframe, total_rows):
    """
    Deletes columns from a DataFrame if the number of non-null values
    is less than (total_rows - 10% of total_rows).

    Args:
        dataframe (pd.DataFrame): The input DataFrame.
        total_rows (int): The total number of rows in the DataFrame (X).

    Returns:
        pd.DataFrame: The DataFrame with sparse columns removed.
    """
    threshold = total_rows - (0.10 * total_rows)
    columns_to_drop = []

    for column in dataframe.columns:
        if dataframe[column].count() < threshold:
            columns_to_drop.append(column)

    if columns_to_drop:
        print(f"Dropping columns due to sparsity: {columns_to_drop}")
        dataframe.drop(columns=columns_to_drop, inplace=True)
    else:
        print("No columns found to drop based on the sparsity criteria.")

    return dataframe

# Apply the function to your DataFrame df, using X as the total number of rows
df = delete_sparse_columns(df, X)

# Display the info of the modified DataFrame to see the changes
df.info()

Dropping columns due to sparsity: ['OBJECTID', 'Station_name']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1958 entries, 0 to 1957
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Station_address  1958 non-null   object 
 1   Operator         1958 non-null   object 
 2   Number_of_plugs  1958 non-null   int64  
 3   Charger_Type     1958 non-null   object 
 4   Charger_rating   1958 non-null   object 
 5   Latitude         1958 non-null   float64
 6   Longitude        1958 non-null   float64
 7   LGANAME          1837 non-null   object 
 8   PCODE            1837 non-null   object 
 9   Source           1837 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 153.1+ KB


# Check Missing Values in Column

In [362]:
df.isnull().sum()

,0
Station_address,0
Operator,0
Number_of_plugs,0
Charger_Type,0
Charger_rating,0
Latitude,0
Longitude,0
LGANAME,121
PCODE,121
Source,121


# Structuring
1. Rename the columns for better understanding
2. Merge Latitute and Longitude column in to one column Location

In [363]:
df = df.rename(columns={
    "LGANAME": "Council",
    "PCODE": "Postcode",
    "Charger_rating": "Power Per Plug(kW)",
    "Number_of_plugs":"Total Plugs","Charger_Type":"Charger Type"
})

In [364]:
df['Location'] = df['Latitude'].astype(str) + ', ' + df['Longitude'].astype(str)
print(df.head())

                                Station_address   Operator  Total Plugs  \
0                          , Muswellbrook, 2333       EVUp            2   
1               01 Wallgrove Road, Sydney, 2766         BP            4   
2  1 - 7 Ross St, Wilcannia NSW 2836, Australia       NRMA            4   
3                    1 Balfour St, Sydney, 2070  Chargefox            7   
4                     1 Bay Ln, Byron Bay, 2481      Tesla            2   

  Charger Type Power Per Plug(kW)   Latitude   Longitude  \
0           AC              22 kW -32.262242  150.890139   
1           DC             150 kW -33.811004  150.849597   
2           DC              50 kW -30.511874  151.669395   
3           AC              22 kW -33.774101  151.167035   
4           AC              19 kW -28.641819  153.613633   

                         Council Postcode                         Source  \
0     Muswellbrook Shire Council     2333  Existing Destination Chargers   
1         Blacktown City Council    

# EV Insights
Show EV Station Markers on Map










In [365]:
import folium

m = folium.Map(location=[df["Latitude"].mean(), df["Longitude"].mean()], zoom_start=10)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=4
    ).add_to(m)

m

Total Ev Working Chargers

In [366]:
ac_chargers = df[df['Charger Type'] == 'AC'].shape[0]
dc_chargers = df[df['Charger Type'] == 'DC'].shape[0]

print(f"Total AC Chargers: {ac_chargers}")
print(f"Total DC Chargers: {dc_chargers}")
print(f"Total Working Chargers (AC & DC): {ac_chargers + dc_chargers}")

Total AC Chargers: 1427
Total DC Chargers: 433
Total Working Chargers (AC & DC): 1860


Upcoming Chargers


In [367]:
upcoming = df[df['Charger Type'] == 'Upcoming'].shape[0]
print(f"Total Upcoming Chargers: {upcoming}")

Total Upcoming Chargers: 98


In [368]:
df["Charger Type"].value_counts()

,count
Charger Type,
AC,1427
DC,433
Upcoming,98


AC chargers are slow & DC chargers are fast
***bold text***


Result: Slow Chargers are more than Fast Chargers


# Council EV Counts


1.   Council with Maximum EV Chargers & its count
2.   Council with Minimum EV Chargers & its count



In [369]:
council_chargers = df.groupby('Council')['Total Plugs'].sum().reset_index()
most_chargers_council = council_chargers.loc[council_chargers['Total Plugs'].idxmax()]

print(f"The council with the most chargers is: {most_chargers_council['Council']} with {most_chargers_council['Total Plugs']} total plugs.")

The council with the most chargers is: Sydney, Council of the City of with 249 total plugs.


In [370]:
minimum_chargers_council = council_chargers.loc[council_chargers['Total Plugs'].idxmin()]

print(f"The council with the minimum chargers is: {minimum_chargers_council['Council']} with {minimum_chargers_council['Total Plugs']} total plugs.")

The council with the minimum chargers is: Bland Shire Council with 2 total plugs.


Top 3 Council with most DC Fast Chargers

In [371]:
dc_chargers_by_council = df[df['Charger Type'] == 'DC'].groupby('Council')['Total Plugs'].sum().reset_index()
top_3_dc_councils = dc_chargers_by_council.sort_values(by='Total Plugs', ascending=False).head(3)

print("Top 3 Councils with Most DC Fast Chargers:")
print(top_3_dc_councils)

Top 3 Councils with Most DC Fast Chargers:
                           Council  Total Plugs
8           Blacktown City Council           67
77           Randwick City Council           66
88  Sydney, Council of the City of           57


Top 3 Council with most AC slow Chargers

In [372]:
ac_chargers_by_council = df[df['Charger Type'] == 'AC'].groupby('Council')['Total Plugs'].sum().reset_index()
top_3_ac_councils = ac_chargers_by_council.sort_values(by='Total Plugs', ascending=False).head(3)

print("Top 3 Councils with Most AC Slow Chargers:")
print(top_3_ac_councils)

Top 3 Councils with Most AC Slow Chargers:
                           Council  Total Plugs
97  Sydney, Council of the City of          192
49              Inner West Council          175
75          Newcastle City Council          112


PostCode EV Chargers Count


1.   Post Code with Maximum EV Chargers
2.   Post Code with Minimum EV Chargers


In [373]:
postcode_chargers = df.groupby('Postcode')['Total Plugs'].sum().reset_index()

max_chargers_postcode = postcode_chargers.loc[postcode_chargers['Total Plugs'].idxmax()]
min_chargers_postcode = postcode_chargers.loc[postcode_chargers['Total Plugs'].idxmin()]

print(f"The postcode with the maximum chargers is: {max_chargers_postcode['Postcode']} with {max_chargers_postcode['Total Plugs']} total plugs.")
print(f"The postcode with the minimum chargers is: {min_chargers_postcode['Postcode']} with {min_chargers_postcode['Total Plugs']} total plugs.")

The postcode with the maximum chargers is: 2000 with 96 total plugs.
The postcode with the minimum chargers is: 2069 with 1 total plugs.


Post Code with Most DC Fast Chargers

In [374]:
dc_chargers_by_postcode = df[df['Charger Type'] == 'DC'].groupby('Postcode')['Total Plugs'].sum().reset_index()
max_dc_chargers_postcode = dc_chargers_by_postcode.loc[dc_chargers_by_postcode['Total Plugs'].idxmax()]

print(f"The postcode with the most DC fast chargers is: {max_dc_chargers_postcode['Postcode']} with {max_dc_chargers_postcode['Total Plugs']} total plugs.")

The postcode with the most DC fast chargers is: 2580 with 38 total plugs.


Post Code with Most AC Slow
Chargers

In [375]:
ac_chargers_by_postcode = df[df['Charger Type'] == 'AC'].groupby('Postcode')['Total Plugs'].sum().reset_index()
max_ac_chargers_postcode = ac_chargers_by_postcode.loc[ac_chargers_by_postcode['Total Plugs'].idxmax()]

print(f"The postcode with the most AC slow chargers is: {max_ac_chargers_postcode['Postcode']} with {max_ac_chargers_postcode['Total Plugs']} total plugs.")

The postcode with the most AC slow chargers is: 2000 with 94 total plugs.


# Operator Market Share

Top 3 Market Share holder in Ev Chargers


In [376]:
operator_counts = df['Operator'].value_counts().reset_index()
operator_counts.columns = ['Operator', 'Charger Count']
top_3_operators = operator_counts.head(3)

print("Top 3 Operators by Charger Count:")
print(top_3_operators)

Top 3 Operators by Charger Count:
    Operator  Charger Count
0   Exploren            301
1      Tesla            260
2  Chargefox            254


Top 3 Operator with most DC Fast Chargers

In [377]:
dc_chargers_df = df[df['Charger Type'] == 'DC']
dc_operator_counts = dc_chargers_df['Operator'].value_counts().reset_index()
dc_operator_counts.columns = ['Operator', 'DC Charger Count']
top_3_dc_operators = dc_operator_counts.head(3)

print("Top 3 Operators with Most DC Fast Chargers:")
print(top_3_dc_operators)

Top 3 Operators with Most DC Fast Chargers:
  Operator  DC Charger Count
0     Evie                84
1     NRMA                67
2    Tesla                54


Top 3 Operator with most AC Slow Chargers

In [378]:
ac_chargers_df = df[df['Charger Type'] == 'AC']
ac_operator_counts = ac_chargers_df['Operator'].value_counts().reset_index()
ac_operator_counts.columns = ['Operator', 'AC Charger Count']
top_3_ac_operators = ac_operator_counts.head(3)

print("Top 3 Operators with Most AC Slow Chargers:")
print(top_3_ac_operators)

Top 3 Operators with Most AC Slow Chargers:
        Operator  AC Charger Count
0       Exploren               278
1  Non-networked               217
2          Tesla               206


# Charging Capacity Analysis



In [379]:
# Clean and convert 'Power Per Plug(kW)' to numeric
df['Power Per Plug(kW)'] = df['Power Per Plug(kW)'].str.replace(' kW', '', regex=False)
df['Power Per Plug(kW)'] = pd.to_numeric(df['Power Per Plug(kW)'], errors='coerce')

# Calculate Total Charging Capacity
df['Charging_Capacity (kW)'] = df['Total Plugs'] * df['Power Per Plug(kW)']

display(df[['Total Plugs', 'Power Per Plug(kW)', 'Charging_Capacity (kW)']].head())


,Total Plugs,Power Per Plug(kW),Charging_Capacity (kW)
0,2,22.0,44.0
1,4,150.0,600.0
2,4,50.0,200.0
3,7,22.0,154.0
4,2,19.0,38.0


Total Charging Capacity in NSW

In [380]:
total_charging_capacity = df['Charging_Capacity (kW)'].sum()
print(f"Total Charging Capacity in NSW: {total_charging_capacity:.2f} kW")

Total Charging Capacity in NSW: 236274.00 kW


Charging station with maximum plugs

In [381]:
max_plugs_station = df.sort_values(by='Total Plugs', ascending=False).iloc[0]

print("Charging Station with Maximum Plugs:")
print(max_plugs_station[['Station_address', 'Operator', 'Total Plugs', 'Charger Type', 'Power Per Plug(kW)']])

Charging Station with Maximum Plugs:
Station_address       2 Conferta Ave Tallawong NSW 2762
Operator                                  Non-networked
Total Plugs                                          35
Charger Type                                         AC
Power Per Plug(kW)                                  NaN
Name: 1172, dtype: object


Charging Station with Maximum Charging Capacity (kW)

In [382]:
max_capacity_station = df.sort_values(by='Charging_Capacity (kW)', ascending=False).iloc[0]

print("Charging Station with Maximum Charging Capacity:")
print(max_capacity_station[['Station_address', 'Operator', 'Total Plugs', 'Charger Type', 'Power Per Plug(kW)', 'Charging_Capacity (kW)']])

Charging Station with Maximum Charging Capacity:
Station_address           M4 Motorway Westbound, Sydney, 2766
Operator                                                Ampol
Total Plugs                                                10
Charger Type                                               DC
Power Per Plug(kW)                                      400.0
Charging_Capacity (kW)                                 4000.0
Name: 898, dtype: object


List Of Postcode By Number of EV Plugs

In [383]:
df.groupby("Postcode")["Total Plugs"].sum().sort_values(ascending=False)

,Total Plugs
Postcode,
2000,96
2320,95
2444,77
2800,70
2065,62
...,...
2069,1
2769,1
2804,1


List Of Postcode By EV Charging Capacity (KW)

In [384]:
df.groupby("Postcode")["Charging_Capacity (kW)"].sum().sort_values(ascending=False)

,Charging_Capacity (kW)
Postcode,
2766,8580.0
2444,5744.0
2580,5727.0
2450,5139.0
2324,4920.0
...,...
2732,0.0
2798,0.0
2792,0.0


Gaps

Average Distance Between Chargers

In [385]:
import numpy as np

# Earth's radius in kilometers
R = 6371

# Extract latitude and longitude
latitudes = df['Latitude'].values
longitudes = df['Longitude'].values

# Convert to radians
latitudes_rad = np.radians(latitudes)
longitudes_rad = np.radians(longitudes)

distances = []

# Calculate pairwise distances using Haversine formula
num_stations = len(latitudes_rad)
for i in range(num_stations):
    for j in range(i + 1, num_stations): # Avoid duplicate pairs and self-distance
        lat1, lon1 = latitudes_rad[i], longitudes_rad[i]
        lat2, lon2 = latitudes_rad[j], longitudes_rad[j]

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        distance = R * c
        distances.append(distance)

if distances:
    average_distance = np.mean(distances)
    print(f"The average distance between charging stations is: {average_distance:.2f} km")
else:
    print("Not enough stations to calculate an average distance.")

The average distance between charging stations is: 247.66 km


In [386]:
df.to_csv("ev_data_clean.csv", index=False)